In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# MINI PROJECT — DAY 11
# Employee Analytics Pipeline
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
'''
Goal:
Build an employee analytics pipeline
using BOTH Pandas and PySpark.
Run the same operations in both.
Compare the results.
This is how real data engineers validate
a PySpark migration from Pandas.

---

DATASET

You will use two files:

cat > employees.csv << EOF
emp_id,name,dept,salary,years,city
1,Alice,Engineering,85000,5,New York
2,Bob,Marketing,62000,3,Chicago
3,Carol,Engineering,92000,8,New York
4,David,HR,55000,2,Dallas
5,Eva,Finance,78000,6,New York
6,Frank,Engineering,88000,7,Chicago
7,Grace,Finance,67000,4,Dallas
8,Henry,HR,71000,5,New York
9,Ian,Marketing,58000,2,Chicago
10,Jane,Engineering,95000,9,New York
EOF

cat > departments.csv << EOF
dept,budget,location
Engineering,1500000,New York
Marketing,800000,Chicago
Finance,600000,New York
HR,400000,Dallas
Legal,350000,New York
EOF
---

STEP 1 - Linux Setup

mkdir employee_pipeline
cd employee_pipeline
mkdir raw processed reports

Create both CSV files in raw/.
Verify with: wc -l raw/*.csv

---

STEP 2 - Pandas Pipeline

Create: processed/pandas_analysis.py

Task 1 — Load and profile:
- Read both CSVs.
- Print shape, dtypes, null counts.
- Print describe() for employees.

Task 2 — Clean and enrich employees:
- Strip whitespace from string columns.
- Add salary_band using apply():
  >= 85000 → "Senior"
  >= 65000 → "Mid"
  below    → "Junior"
- Add experience_band using apply():
  years >= 7 → "Veteran"
  years >= 4 → "Experienced"
  below 4    → "Fresher"

Task 3 — Merge with departments:
- Left join employees with departments
  on dept column.
- Keep: name, dept, salary, years, city,
        budget, location, salary_band,
        experience_band.
- Save to processed/enriched_employees.csv

Task 4 — Analysis (save each result):
- Dept summary: headcount, avg_salary,
  total_salary, budget, variance
  (budget minus total_salary).
  Save to reports/dept_summary.csv

- Salary band distribution:
  count per band.
  Save to reports/band_distribution.csv

- Top 3 earners per department.
  Save to reports/top_earners.csv

- City-wise headcount and avg salary.
  Save to reports/city_summary.csv

---

STEP 3 - PySpark Pipeline

Create: processed/spark_analysis.py

Repeat ALL four tasks from Step 2
using PySpark.

Use:
- spark.read.csv() for loading
- withColumn() + when() for new columns
- df.join() for merging
- groupBy().agg() for analysis
- df.write.csv() for saving output

Save PySpark outputs to:
reports/spark_dept_summary.csv
reports/spark_band_distribution.csv
reports/spark_top_earners.csv
reports/spark_city_summary.csv

---

STEP 4 - SQL Verification

Run these four queries on freesql.com.
Create the employees and departments
tables with the same 10-row dataset.

1. Dept summary with budget variance.
2. Salary band distribution using
   conditional aggregation.
3. Top earner per department using
   subquery.
4. City-wise headcount and avg salary.

All four SQL results must match both
the Pandas and PySpark outputs.
---

STEP 5 - Comparison check

Create: reports/comparison.py

Write a function that:
- Loads pandas output CSV
- Loads pyspark output CSV
- Compares both DataFrames using
  df1.equals(df2) or a row-by-row check
- Prints MATCH or MISMATCH for each report

This is called a reconciliation check.
In production pipelines this step
confirms that a PySpark migration
produces identical results to the
original Pandas pipeline.

---

STEP 6 - Archive

tar -czvf employee_pipeline_day11.tar.gz
employee_pipeline/

Verify:
tar -tvf employee_pipeline_day11.tar.gz

---

End Goal:
You have run the exact same analytics
pipeline in three ways — Pandas, PySpark,
and SQL — and verified all three produce
identical results.

This is the core skill of a data engineer:
not just knowing one tool, but knowing
how the same operation looks across
all layers of the stack.

Pandas   → local processing, quick analysis
PySpark  → distributed, production scale
SQL      → querying, reporting, verification

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
DAILY TARGETS
60 minutes  -- Solve exercises
20 minutes  -- Review and discuss in group
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
'''

In [ ]:
# STEP 4 - SQL Verification

# DATASET
'''
CREATE TABLE EMP(
    EMP_ID INT, 
    NAME VARCHAR2(50), 
    DEPT VARCHAR2(50), 
    SALARY INT, 
    YEARS INT, 
    CITY VARCHAR2(50));

INSERT INTO EMP VALUES (1, 'Alice', 'Engineering', 85000, 5, 'New York');
INSERT INTO EMP VALUES (2, 'Bob', 'Marketing', 62000, 3, 'Chicago');
INSERT INTO EMP VALUES (3, 'Carol', 'Engineering', 92000, 8, 'New York');
INSERT INTO EMP VALUES (4, 'David', 'HR', 55000, 2, 'Dallas');
INSERT INTO EMP VALUES (5, 'Eva', 'Finance', 78000, 6, 'New York');
INSERT INTO EMP VALUES (6, 'Frank', 'Engineering', 88000, 7, 'Chicago');
INSERT INTO EMP VALUES (7, 'Grace', 'Finance', 67000, 4, 'Dallas');
INSERT INTO EMP VALUES (8, 'Henry', 'HR', 71000, 5, 'New York');
INSERT INTO EMP VALUES (9, 'Ian', 'Marketing', 58000, 2, 'Chicago');
INSERT INTO EMP VALUES (10, 'Jane', 'Engineering', 95000, 9, 'New York');

SELECT * FROM EMP;
'''
'''

CREATE TABLE DEPT(DEPT VARCHAR2(50), BUDGET INT, LOCATION VARCHAR2(50));

INSERT INTO DEPT VALUES('Engineering', 1500000, 'New York');
INSERT INTO DEPT VALUES('Marketing', 800000, 'Chicago');
INSERT INTO DEPT VALUES('Finance', 600000, 'New York');
INSERT INTO DEPT VALUES('HR', 400000, 'Dallas');
INSERT INTO DEPT VALUES('Legal', 350000, 'New York');

SELECT * FROM DEPT;
'''

# 1 
SELECT 
    E.DEPT,
    COUNT(*) AS EMP_COUNT,
    AVG(SALARY) AS AVG_SALARY,
    SUM(E.SALARY) AS TOTAL_SALARY,
    D.BUDGET,
    (D.BUDGET - SUM(E.SALARY)) AS VARIANCE
FROM EMP E
LEFT JOIN DEPT D
ON E.DEPT = D.DEPT
GROUP BY E.DEPT,
D.BUDGET;

# 2
SELECT
    CASE
        WHEN SALARY >= 85000 THEN 'Senior'
        WHEN SALARY >= 65000 THEN 'Mid'
        ELSE 'Junior'
    END AS SALARY_BAND,

    COUNT(*) AS HEADCOUNT

    FROM EMP

    GROUP BY
    CASE
        WHEN SALARY >= 85000 THEN 'Senior'
        WHEN SALARY >= 65000 THEN 'Mid'
        ELSE 'Junior'
    END;

#  4
SELECT 
    CITY,
    COUNT(*) AS EMP_COUNT,
    ROUND(AVG(SALARY), 2) AS AVG_SALARY
FROM EMP
GROUP BY CITY;